In [7]:
import pandas as pd
import numpy as np
import re

bts_history = pd.read_excel('bts_history_2.xlsx')
promo = pd.read_excel('promo_2.xlsx')
pricing_plan = pd.read_excel('pricing_plan_2026_2.xlsx')

def convert_weight_to_kg(weight_str):
    weight_str = str(weight_str).strip().lower()
    if 'kg' in weight_str:
        return float(re.sub(r'[^\d\.]', '', weight_str))
    elif 'g' in weight_str:
        return float(re.sub(r'[^\d\.]', '', weight_str)) / 1000.0
    else:
        return float(weight_str)

def parse_period_name(period_name):
    parts = period_name.strip().split()
    if len(parts) != 2:
        raise ValueError(f"Неверный формат периода: {period_name}")
    period_in_year = int(parts[0][1:])
    year = int(parts[1])
    return year, period_in_year

def add_period_columns(df, period_col='period_name'):
    df[['year', 'period_in_year']] = df[period_col].apply(
        lambda x: pd.Series(parse_period_name(x))
    )
    df['period_abs'] = (df['year'] - 2021) * 13 + (df['period_in_year'] - 1)
    return df

bts_history['weight_kg'] = bts_history['weight_kg'].apply(convert_weight_to_kg)
bts_history = add_period_columns(bts_history, 'period_name')

promo['weight_kg'] = promo['weight_kg'].apply(convert_weight_to_kg)
promo = add_period_columns(promo, 'period_name')
if 'promo_depth' not in promo.columns:
    promo['promo_depth'] = 0

pricing_plan['weight_kg'] = pricing_plan['weight_kg'].apply(convert_weight_to_kg)
pricing_plan = add_period_columns(pricing_plan, 'period_name')
assert (pricing_plan['year'] == 2026).all(), "pricing_plan содержит не только 2026 год"

train_data = bts_history.merge(
    promo[['client_id', 'brand', 'technology', 'weight_kg', 'period_abs', 'promo_days', 'promo_depth']],
    on=['client_id', 'brand', 'technology', 'weight_kg', 'period_abs'],
    how='left'
)
train_data['promo_days'] = train_data['promo_days'].fillna(0)
train_data['promo_depth'] = train_data['promo_depth'].fillna(0)
train_data = train_data[train_data['year'] <= 2025].copy()

train_data = train_data.sort_values(['client_id', 'brand', 'technology', 'weight_kg', 'period_abs'])
group_cols = ['client_id', 'brand', 'technology', 'weight_kg']

train_data['tons_lag1'] = train_data.groupby(group_cols)['tons'].shift(1)
train_data['tons_lag13'] = train_data.groupby(group_cols)['tons'].shift(13)
train_data['tons_ma3'] = train_data.groupby(group_cols)['tons'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)
train_data_clean = train_data.dropna(subset=['tons_lag1', 'tons_lag13']).copy()


unique_keys = train_data[group_cols].drop_duplicates()

periods_2026 = []
for p in range(1, 14):
    periods_2026.append((2026, p, (2026-2021)*13 + (p-1)))
periods_df = pd.DataFrame(periods_2026, columns=['year', 'period_in_year', 'period_abs'])

forecast_base = unique_keys.merge(periods_df, how='cross')
forecast_base['period_name'] = 'P' + forecast_base['period_in_year'].astype(str) + ' ' + forecast_base['year'].astype(str)

forecast_base = forecast_base.merge(
    pricing_plan[group_cols + ['period_abs', 'nestle_effect_pct', 'mars_effect_pct']],
    on=group_cols + ['period_abs'],
    how='left'
)
forecast_base[['nestle_effect_pct', 'mars_effect_pct']] = forecast_base[['nestle_effect_pct', 'mars_effect_pct']].fillna(0)

promo_2026 = promo[promo['year'] == 2026].copy()
forecast_base = forecast_base.merge(
    promo_2026[group_cols + ['period_abs', 'promo_days', 'promo_depth']],
    on=group_cols + ['period_abs'],
    how='left'
)
forecast_base[['promo_days', 'promo_depth']] = forecast_base[['promo_days', 'promo_depth']].fillna(0)


# tons_lag13
sales_2025 = train_data[train_data['year'] == 2025][group_cols + ['period_in_year', 'tons']].rename(columns={'tons': 'tons_lag13'})
forecast_base = forecast_base.merge(sales_2025, on=group_cols + ['period_in_year'], how='left')
forecast_base['tons_lag13'] = forecast_base['tons_lag13'].fillna(0)

# tons_lag1
last_known = train_data[train_data['year'] == 2025].groupby(group_cols).last().reset_index()
last_known = last_known[group_cols + ['tons']].rename(columns={'tons': 'tons_lag1_p1'})
forecast_base = forecast_base.merge(last_known, on=group_cols, how='left')
forecast_base['tons_lag1'] = np.nan
forecast_base.loc[forecast_base['period_in_year'] == 1, 'tons_lag1'] = forecast_base.loc[forecast_base['period_in_year'] == 1, 'tons_lag1_p1']
forecast_base.drop('tons_lag1_p1', axis=1, inplace=True)

# MA3
last_3_2025 = train_data[train_data['year'] == 2025].groupby(group_cols).tail(3).groupby(group_cols)['tons'].mean().reset_index().rename(columns={'tons': 'tons_ma3_p1'})
forecast_base = forecast_base.merge(last_3_2025, on=group_cols, how='left')
forecast_base['tons_ma3'] = np.nan
forecast_base.loc[forecast_base['period_in_year'] == 1, 'tons_ma3'] = forecast_base.loc[forecast_base['period_in_year'] == 1, 'tons_ma3_p1']
forecast_base.drop('tons_ma3_p1', axis=1, inplace=True)
forecast_base = forecast_base[forecast_base['period_in_year'].between(1, 13)]

train_data_clean.to_csv('train_ready.csv', index=False)
forecast_base.to_csv('forecast_base_2026.csv', index=False)


In [13]:
display(train_data_clean.head(10))

,client_id,brand,technology,weight_kg,period_name,tons,nestle_effect_pct,mars_effect_pct,year,period_in_year,period_abs,promo_days,promo_depth,tons_lag1,tons_lag13,tons_ma3
13,client_1,A,dry,1.5,P1 2022,27.852158,0.0,0.0,2022,1,13,7,14,24.474728,26.660407,24.933234
14,client_1,A,dry,1.5,P2 2022,28.321668,6.1,0.0,2022,2,14,0,0,27.852158,26.346424,25.704376
15,client_1,A,dry,1.5,P3 2022,27.255477,4.4,-4.6,2022,3,15,12,19,28.321668,25.055967,26.882851
16,client_1,A,dry,1.5,P4 2022,25.022489,3.8,-2.8,2022,4,16,0,0,27.255477,24.458197,27.809768
17,client_1,A,dry,1.5,P5 2022,25.649296,6.2,-4.4,2022,5,17,11,11,25.022489,24.880195,26.866545
18,client_1,A,dry,1.5,P6 2022,26.125998,6.0,-3.8,2022,6,18,0,0,25.649296,26.039183,25.975754
19,client_1,A,dry,1.5,P7 2022,25.918788,0.0,-4.6,2022,7,19,7,11,26.125998,25.063794,25.599261
20,client_1,A,dry,1.5,P8 2022,27.028184,0.0,0.0,2022,8,20,0,0,25.918788,26.039045,25.898027
21,client_1,A,dry,1.5,P9 2022,24.829761,0.0,0.0,2022,9,21,0,0,27.028184,26.022964,26.357657
22,client_1,A,dry,1.5,P10 2022,25.402745,0.0,0.0,2022,10,22,0,0,24.829761,25.692329,25.925578


In [15]:
display(forecast_base.head(10))

,client_id,brand,technology,weight_kg,year,period_in_year,period_abs,period_name,nestle_effect_pct,mars_effect_pct,promo_days,promo_depth,tons_lag13,tons_lag1,tons_ma3
0,client_1,A,dry,1.5,2026,1,65,P1 2026,0.0,0.0,13,18,28.500652,27.885734,28.337144
1,client_1,A,dry,1.5,2026,2,66,P2 2026,6.4,0.0,15,23,31.680387,NaN,NaN
2,client_1,A,dry,1.5,2026,3,67,P3 2026,4.8,-2.9,11,11,30.068468,NaN,NaN
3,client_1,A,dry,1.5,2026,4,68,P4 2026,4.0,-4.7,13,17,31.613696,NaN,NaN
4,client_1,A,dry,1.5,2026,5,69,P5 2026,3.7,-4.7,0,0,28.263338,NaN,NaN
5,client_1,A,dry,1.5,2026,6,70,P6 2026,3.1,-3.0,0,0,27.879458,NaN,NaN
6,client_1,A,dry,1.5,2026,7,71,P7 2026,0.0,-4.4,0,0,30.756733,NaN,NaN
7,client_1,A,dry,1.5,2026,8,72,P8 2026,0.0,0.0,0,0,25.707091,NaN,NaN
8,client_1,A,dry,1.5,2026,9,73,P9 2026,0.0,0.0,6,17,28.340280,NaN,NaN
9,client_1,A,dry,1.5,2026,10,74,P10 2026,0.0,0.0,0,0,28.392074,NaN,NaN
